# Razonamiento Avanzado: Chain-of-Thought, Self-Consistency y Extended Thinking

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/llms/13-razonamiento-avanzado.ipynb)

Técnicas de razonamiento para mejorar la precisión en tareas complejas: CoT, Self-Consistency, Tree of Thoughts y Extended Thinking de Claude.

**Contenidos:**
1. Instalación y configuración
2. Chain-of-Thought zero-shot
3. Few-shot CoT con ejemplos
4. Self-Consistency Reasoner
5. Tree of Thoughts simplificado
6. Extended Thinking con Claude Sonnet
7. Comparativa de precisión en problema matemático
8. Conclusiones y guía de selección

In [ ]:
# Instalación
# !pip install anthropic

import os
import re
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import Optional

import anthropic

# Configurar API key
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"
print(f"Cliente Anthropic configurado. Modelo por defecto: {MODEL}")

## 1. Chain-of-Thought Zero-Shot

La forma más simple de CoT: añadir "Piensa paso a paso" al prompt. Sin ejemplos, sin configuración adicional.

In [ ]:
def respuesta_directa(problema: str) -> dict:
    """Respuesta sin razonamiento explícito (baseline)."""
    inicio = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        messages=[{"role": "user", "content": problema}]
    )
    return {
        "tecnica": "Directa",
        "respuesta": response.content[0].text,
        "tokens": response.usage.input_tokens + response.usage.output_tokens,
        "tiempo_s": round(time.time() - inicio, 2)
    }


def cot_zero_shot(problema: str) -> dict:
    """CoT zero-shot: solo añadimos la instrucción de pensar paso a paso."""
    inicio = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{
            "role": "user",
            "content": f"{problema}\n\nPiensa paso a paso antes de dar tu respuesta final."
        }]
    )
    texto = response.content[0].text
    return {
        "tecnica": "CoT Zero-Shot",
        "razonamiento": texto,
        "respuesta": texto.split("\n")[-1],
        "tokens": response.usage.input_tokens + response.usage.output_tokens,
        "tiempo_s": round(time.time() - inicio, 2)
    }


# Problema de prueba
PROBLEMA_1 = """
Una tienda tiene 240 artículos. El lunes vende el 35% del total.
El martes vende el 25% de los artículos que quedaban.
El miércoles recibe una reposición de 50 artículos.
¿Cuántos artículos tiene la tienda al final del miércoles?
"""

r_directa = respuesta_directa(PROBLEMA_1)
r_cot = cot_zero_shot(PROBLEMA_1)

print("=== RESPUESTA DIRECTA ===")
print(r_directa["respuesta"])
print(f"Tokens: {r_directa['tokens']} | Tiempo: {r_directa['tiempo_s']}s")

print("\n=== COT ZERO-SHOT ===")
print(r_cot["razonamiento"])
print(f"Tokens: {r_cot['tokens']} | Tiempo: {r_cot['tiempo_s']}s")

## 2. Few-Shot CoT con Ejemplos

Proporcionamos ejemplos completos de razonamiento para que el modelo imite el formato y nivel de detalle deseado.

In [ ]:
FEW_SHOT_SYSTEM = """Eres un asistente matemático. Resuelves cada problema mostrando
explícitamente cada paso con el formato:

Paso 1: [descripción]
Cálculo: [operación]
Resultado: [valor]
...
RESPUESTA FINAL: [valor con unidades]"""

EJEMPLOS = [
    {
        "role": "user",
        "content": "Juan tiene el triple de canicas que Pedro. Pedro tiene 12. ¿Cuántas tienen entre los dos?"
    },
    {
        "role": "assistant",
        "content": """Paso 1: Calcular canicas de Juan
Cálculo: Juan = 3 × 12 = 36
Resultado: Juan tiene 36 canicas

Paso 2: Sumar canicas de ambos
Cálculo: Total = 36 + 12 = 48
Resultado: 48 canicas

RESPUESTA FINAL: 48 canicas"""
    }
]


def cot_few_shot(problema: str) -> dict:
    """Few-shot CoT: el modelo imita el formato de razonamiento del ejemplo."""
    inicio = time.time()
    messages = EJEMPLOS + [{"role": "user", "content": problema}]
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=FEW_SHOT_SYSTEM,
        messages=messages
    )
    texto = response.content[0].text
    
    # Extraer respuesta final
    match = re.search(r"RESPUESTA FINAL:\s*(.+?)(?:\n|$)", texto, re.IGNORECASE)
    respuesta_final = match.group(1).strip() if match else texto.split("\n")[-1]
    
    return {
        "tecnica": "CoT Few-Shot",
        "razonamiento": texto,
        "respuesta": respuesta_final,
        "tokens": response.usage.input_tokens + response.usage.output_tokens,
        "tiempo_s": round(time.time() - inicio, 2)
    }


r_few_shot = cot_few_shot(PROBLEMA_1)
print("=== COT FEW-SHOT ===")
print(r_few_shot["razonamiento"])
print(f"\nRespuesta extraída: {r_few_shot['respuesta']}")
print(f"Tokens: {r_few_shot['tokens']} | Tiempo: {r_few_shot['tiempo_s']}s")

## 3. Self-Consistency Reasoner

Genera N respuestas independientes con temperatura > 0 y elige la más frecuente por votación mayoritaria.

In [ ]:
@dataclass
class SelfConsistencyReasoner:
    """
    Self-Consistency: genera N respuestas y vota la más frecuente.
    
    Args:
        model: Modelo a usar
        n_muestras: Número de respuestas independientes
        temperatura: Temperatura para la diversidad (recomendado: 0.7-1.0)
    """
    model: str = "claude-haiku-4-5-20251001"
    n_muestras: int = 3
    temperatura: float = 0.8
    _client: anthropic.Anthropic = field(init=False, default_factory=anthropic.Anthropic)
    
    def _generar_respuesta(self, problema: str) -> str:
        response = self._client.messages.create(
            model=self.model,
            max_tokens=400,
            temperature=self.temperatura,
            messages=[{
                "role": "user",
                "content": (
                    f"{problema}\n\n"
                    "Razona paso a paso. Al final escribe:\n"
                    "RESPUESTA: [solo el valor numérico o la opción]"
                )
            }]
        )
        return response.content[0].text
    
    def _extraer_respuesta(self, texto: str) -> Optional[str]:
        match = re.search(r"RESPUESTA:\s*(.+?)(?:\n|$)", texto, re.IGNORECASE)
        if match:
            return match.group(1).strip().lower()
        return None
    
    def razonar(self, problema: str) -> dict:
        """Resuelve con Self-Consistency y devuelve respuesta con métricas."""
        inicio = time.time()
        respuestas_raw = []
        respuestas_extraidas = []
        
        print(f"Generando {self.n_muestras} muestras independientes...", end=" ")
        for i in range(self.n_muestras):
            raw = self._generar_respuesta(problema)
            respuestas_raw.append(raw)
            extraida = self._extraer_respuesta(raw)
            if extraida:
                respuestas_extraidas.append(extraida)
            print(f"[{i+1}]", end=" ", flush=True)
        print("listo")
        
        if not respuestas_extraidas:
            return {"error": "No se pudo extraer ninguna respuesta"}
        
        conteo = Counter(respuestas_extraidas)
        ganadora, votos = conteo.most_common(1)[0]
        confianza = votos / len(respuestas_extraidas)
        
        return {
            "tecnica": "Self-Consistency",
            "respuesta": ganadora,
            "votos": votos,
            "total_muestras": len(respuestas_extraidas),
            "confianza": f"{confianza:.0%}",
            "distribucion": dict(conteo),
            "tiempo_s": round(time.time() - inicio, 2)
        }


# Aplicar Self-Consistency al mismo problema
reasoner = SelfConsistencyReasoner(n_muestras=3, temperatura=0.8)
r_sc = reasoner.razonar(PROBLEMA_1)

print("\n=== SELF-CONSISTENCY ===")
print(f"Respuesta ganadora: {r_sc['respuesta']}")
print(f"Votos: {r_sc['votos']}/{r_sc['total_muestras']}")
print(f"Confianza: {r_sc['confianza']}")
print(f"Distribución: {r_sc['distribucion']}")
print(f"Tiempo: {r_sc['tiempo_s']}s")

## 4. Tree of Thoughts (Simplificado)

Exploración en árbol: expandir → evaluar → podar → elegir el mejor camino.

In [ ]:
@dataclass
class Nodo:
    contenido: str
    puntuacion: float = 0.0
    profundidad: int = 0
    hijos: list = field(default_factory=list)


class TreeOfThoughts:
    """ToT simplificado con BFS y evaluación por LLM."""
    
    def __init__(self, model: str = "claude-haiku-4-5-20251001", n_ramas: int = 2, profundidad_max: int = 2):
        self.client = anthropic.Anthropic()
        self.model = model
        self.n_ramas = n_ramas
        self.profundidad_max = profundidad_max
    
    def _expandir(self, problema: str, camino: str) -> list[str]:
        response = self.client.messages.create(
            model=self.model,
            max_tokens=300,
            messages=[{"role": "user", "content": (
                f"Problema: {problema}\n\n"
                f"Razonamiento actual:\n{camino if camino else '(inicio)'}\n\n"
                f"Genera exactamente {self.n_ramas} posibles pasos siguientes distintos. "
                f"Numera con 'Opción X:'. Sé conciso (1 frase por opción)."
            )}]
        )
        texto = response.content[0].text
        opciones = re.findall(r"Opci[oó]n \d+:\s*(.+?)(?=Opci[oó]n \d+:|$)", texto, re.DOTALL | re.IGNORECASE)
        return [op.strip() for op in opciones[:self.n_ramas]]
    
    def _evaluar(self, problema: str, camino: str) -> float:
        response = self.client.messages.create(
            model=self.model,
            max_tokens=10,
            messages=[{"role": "user", "content": (
                f"Problema: {problema}\n\nCamino:\n{camino}\n\n"
                f"Puntúa 0-10 qué tan prometedor es. Solo el número."
            )}]
        )
        try:
            return float(response.content[0].text.strip().split()[0])
        except (ValueError, IndexError):
            return 5.0
    
    def resolver(self, problema: str) -> dict:
        inicio = time.time()
        raiz = Nodo(contenido="", profundidad=0)
        frontera = [raiz]
        mejor_nodo = raiz
        mejor_puntuacion = 0.0
        
        for nivel in range(self.profundidad_max):
            siguiente = []
            for nodo in frontera:
                expansiones = self._expandir(problema, nodo.contenido)
                for exp in expansiones:
                    nuevo_camino = f"{nodo.contenido}\n{exp}" if nodo.contenido else exp
                    punt = self._evaluar(problema, nuevo_camino)
                    hijo = Nodo(contenido=nuevo_camino, puntuacion=punt, profundidad=nivel+1)
                    nodo.hijos.append(hijo)
                    siguiente.append(hijo)
                    if punt > mejor_puntuacion:
                        mejor_puntuacion = punt
                        mejor_nodo = hijo
            siguiente.sort(key=lambda n: n.puntuacion, reverse=True)
            frontera = siguiente[:self.n_ramas]
            print(f"Nivel {nivel+1}: mejor puntuación = {mejor_puntuacion:.1f}")
        
        return {
            "tecnica": "Tree of Thoughts",
            "mejor_camino": mejor_nodo.contenido,
            "puntuacion": mejor_puntuacion,
            "tiempo_s": round(time.time() - inicio, 2)
        }


# ToT para un problema de planificación (más adecuado que aritmética)
PROBLEMA_PLANIFICACION = """Tengo 3 horas libres esta tarde. Necesito:
1. Hacer ejercicio (mínimo 45 min)
2. Preparar una presentación para mañana (mínimo 1h)
3. Llamar a mi médico para cita (15 min)
¿Cuál es el mejor orden y distribución del tiempo?"""

tot = TreeOfThoughts(n_ramas=2, profundidad_max=2)
r_tot = tot.resolver(PROBLEMA_PLANIFICACION)

print("\n=== TREE OF THOUGHTS ===")
print("Mejor camino de razonamiento:")
print(r_tot["mejor_camino"])
print(f"\nPuntuación: {r_tot['puntuacion']}/10")
print(f"Tiempo: {r_tot['tiempo_s']}s")

## 5. Extended Thinking de Claude

Claude genera tokens de razonamiento interno antes de producir la respuesta. Requiere `claude-sonnet-4-5` o superior.

In [ ]:
def extended_thinking(problema: str, budget_tokens: int = 5000) -> dict:
    """
    Extended Thinking de Claude.
    
    Nota: Requiere claude-sonnet-4-5-20251001 o claude-opus-4-5.
    No disponible en Haiku.
    
    Args:
        problema: Tarea a resolver
        budget_tokens: Tokens máximos para el bloque thinking (mínimo 1024)
    """
    MODEL_THINKING = "claude-sonnet-4-5-20251001"
    inicio = time.time()
    
    response = client.messages.create(
        model=MODEL_THINKING,
        max_tokens=budget_tokens + 1024,
        thinking={
            "type": "enabled",
            "budget_tokens": budget_tokens
        },
        messages=[{"role": "user", "content": problema}]
    )
    
    thinking_text = ""
    respuesta_text = ""
    
    for block in response.content:
        if block.type == "thinking":
            thinking_text = block.thinking
        elif block.type == "text":
            respuesta_text = block.text
    
    return {
        "tecnica": "Extended Thinking",
        "thinking_preview": thinking_text[:400] + "..." if len(thinking_text) > 400 else thinking_text,
        "respuesta": respuesta_text,
        "tokens_input": response.usage.input_tokens,
        "tokens_output": response.usage.output_tokens,
        "tiempo_s": round(time.time() - inicio, 2)
    }


# Problema matemático complejo — ideal para Extended Thinking
PROBLEMA_COMPLEJO = """
Cinco personas (A, B, C, D, E) se sientan aleatoriamente en una mesa circular con 5 sillas.
¿Cuál es la probabilidad de que A y B sean vecinos (adyacentes)?
Expresa el resultado como fracción simplificada y como porcentaje.
"""

# Descomenta para ejecutar (requiere Sonnet, más costoso que Haiku)
# r_et = extended_thinking(PROBLEMA_COMPLEJO, budget_tokens=3000)
# print("=== EXTENDED THINKING ===")
# print(f"Thinking (preview): {r_et['thinking_preview']}")
# print(f"\nRespuesta: {r_et['respuesta']}")
# print(f"Tokens: {r_et['tokens_input']} entrada + {r_et['tokens_output']} salida")
# print(f"Tiempo: {r_et['tiempo_s']}s")

print("Extended Thinking disponible en claude-sonnet-4-5-20251001 y superior.")
print("Descomenta el bloque anterior para ejecutar (coste mayor que Haiku).")
print("\nRespuesta esperada al problema de probabilidad: 2/5 = 40%")

## 6. Comparativa de Precisión en Problema Matemático

Evaluamos todas las técnicas en el mismo problema y comparamos: precisión, tokens y tiempo.

In [ ]:
# Problema con respuesta verificable
PROBLEMA_BENCHMARK = """
Un tren sale de Madrid a las 08:00 a 120 km/h hacia Barcelona (620 km).
Otro tren sale de Barcelona a las 09:30 a 90 km/h hacia Madrid.
¿A qué hora se cruzan? ¿A qué distancia de Madrid?
"""

# Respuesta correcta para verificación:
# Distancia recorrida por Madrid en t horas desde 08:00:
# Tren Madrid: 120*t
# Tren Barcelona sale a 09:30 (t=1.5h): 90*(t-1.5)
# 120*t + 90*(t-1.5) = 620
# 210t = 620 + 135 = 755 → t = 755/210 ≈ 3.595h desde 08:00
# 08:00 + 3h 35.7min ≈ 11:36
# Distancia Madrid: 120 * 3.595 ≈ 431 km
RESPUESTA_CORRECTA_KM = 431

def evaluar_precision(respuesta: str, valor_correcto: int, tolerancia: int = 5) -> bool:
    """Extrae el número más relevante de la respuesta y verifica si es correcto."""
    numeros = re.findall(r'\b(\d{3,4})\b', respuesta)
    for n in numeros:
        if abs(int(n) - valor_correcto) <= tolerancia:
            return True
    return False


print("Ejecutando comparativa...\n")
resultados = []

# 1. Respuesta directa
r = respuesta_directa(PROBLEMA_BENCHMARK)
r["correcto"] = evaluar_precision(r["respuesta"], RESPUESTA_CORRECTA_KM)
resultados.append(r)
print(f"[1/3] Directa completada")

# 2. CoT Few-Shot
r = cot_few_shot(PROBLEMA_BENCHMARK)
r["correcto"] = evaluar_precision(r["respuesta"], RESPUESTA_CORRECTA_KM)
resultados.append(r)
print(f"[2/3] CoT Few-Shot completada")

# 3. Self-Consistency (3 muestras)
sc = SelfConsistencyReasoner(n_muestras=3, temperatura=0.8)
r = sc.razonar(PROBLEMA_BENCHMARK)
r["correcto"] = evaluar_precision(r["respuesta"], RESPUESTA_CORRECTA_KM)
resultados.append(r)
print(f"[3/3] Self-Consistency completada")

# Mostrar resultados
print("\n" + "="*65)
print(f"{'Técnica':<20} {'¿Correcto?':<12} {'Tokens':<10} {'Tiempo':<10}")
print("-"*65)
for r in resultados:
    correcto = "✓ Sí" if r.get("correcto") else "✗ No"
    tokens = r.get("tokens", "N/A")
    tiempo = f"{r.get('tiempo_s', '?')}s"
    print(f"{r['tecnica']:<20} {correcto:<12} {str(tokens):<10} {tiempo:<10}")
print("="*65)
print(f"Respuesta correcta: ~{RESPUESTA_CORRECTA_KM} km de Madrid (≈ 11:35-11:37)")

## 7. Conclusiones y Guía de Selección

In [ ]:
GUIA = """
╔══════════════════════════════════════════════════════════════╗
║           GUÍA DE SELECCIÓN DE TÉCNICAS DE RAZONAMIENTO      ║
╠══════════════════════════════════════════════════════════════╣
║ Situación                      → Técnica recomendada         ║
╠══════════════════════════════════════════════════════════════╣
║ Chat en tiempo real            → Zero-shot CoT               ║
║ Tarea simple o QA              → Respuesta directa           ║
║ Matemáticas/lógica en batch    → Self-Consistency (N=3-5)    ║
║ Planificación o estrategia     → Tree of Thoughts            ║
║ Análisis legal/médico/técnico  → Extended Thinking           ║
║ Formato estructurado requerido → Few-shot CoT                ║
╠══════════════════════════════════════════════════════════════╣
║              COMPARATIVA LATENCIA / COSTE                    ║
╠══════════════════════════════════════════════════════════════╣
║ Respuesta directa    │ 1×  coste │ 1×  tiempo │ Precisión: ⭐ ║
║ Zero-shot CoT        │ 1.5× coste│ 1.5× tiempo│ Precisión: ⭐⭐║
║ Few-shot CoT         │ 2×  coste │ 1.5× tiempo│ Precisión: ⭐⭐⭐║
║ Self-Consistency(N=5)│ 5×  coste │ 5×  tiempo │ Precisión: ⭐⭐⭐⭐║
║ Extended Thinking    │ 3-8× coste│ 2-4× tiempo│ Precisión: ⭐⭐⭐⭐⭐║
╚══════════════════════════════════════════════════════════════╝
"""
print(GUIA)

print("Regla práctica:")
print("  - Precisión insuficiente con CoT → prueba Self-Consistency")
print("  - Self-Consistency sigue fallando → prueba Extended Thinking")
print("  - El problema tiene múltiples estrategias → Tree of Thoughts")
print("  - Temperatura 0 con Self-Consistency = no sirve (sin diversidad)")